# 02 — Run the toy ACORN pipeline

**Student notebook · 45–60 minutes · CPU only**

Follow one tiny dataset through ACORN training, inference, and evaluation. The model is deliberately treated as a black box here; tutorial 03 opens that box.

## How to use this notebook

Run cells in order with **Shift+Enter**. The three ACORN commands train only a tiny one-step GNN and normally finish within a few minutes on CPU. Fill all six exercise cells and use their assertions as immediate feedback.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import tempfile

import pandas as pd
import torch
import yaml

ROOT = Path.cwd()
if not (ROOT / "vendor" / "acorn").is_dir():
    ROOT = ROOT.parent.resolve()
assert (ROOT / "vendor" / "acorn" / "acorn").is_dir(), "Initialize the ACORN submodule first"

DATA = ROOT / "tutorial_data" / "edge_classifier"
runtime = tempfile.TemporaryDirectory(prefix="acorn_tutorial_02_")
RUNTIME = Path(runtime.name)
STAGE_DIR = RUNTIME / "interaction_gnn"

ACORN_ENV = os.environ.copy()
ACORN_ENV["PYTHONPATH"] = str(ROOT / "vendor" / "acorn") + os.pathsep + ACORN_ENV.get("PYTHONPATH", "")
ACORN_ENV["MPLCONFIGDIR"] = str(RUNTIME / "matplotlib")

def run_acorn(operation, config_path):
    command = [sys.executable, "-m", "acorn.core.entrypoint_stage", operation, str(config_path)]
    result = subprocess.run(command, cwd=ROOT, env=ACORN_ENV, text=True,
                            capture_output=True, timeout=240, check=False)
    lines = (result.stdout + result.stderr).splitlines()
    print("$ acorn", operation, config_path.name)
    print("\n".join(lines[-20:]))
    result.check_returncode()
    return result

print("runtime workspace:", RUNTIME)

## 1. Inputs: split folders of PyG events

ACORN stages exchange `.pyg` event files. This repository bundles eight training, two validation, and two test events. They extend tutorial 01's graph with the metadata required by `EdgeClassifierStage`.

In [ ]:
event_path = None  # TODO: first .pyg path in DATA / "trainset"
event = None       # TODO: torch.load it on CPU with weights_only=False

required = {"hit_x", "hit_r", "hit_phi", "hit_z", "edge_index", "edge_y",
            "track_edges", "track_to_edge_map", "event_id", "config"}
print(event)
assert required <= set(event.keys())
assert event.num_nodes == 8
assert event.edge_index.shape == (2, 12)

## 2. Configuration is the pipeline interface

A short YAML file becomes a Python dictionary. `stage` chooses an ACORN subpackage and `model` chooses a class exported by that stage. `input_dir` reads the preceding stage's artifacts; `stage_dir` receives this stage's logs, checkpoints, plots, and predictions.

In [ ]:
key_meanings = {
    "stage": None,      # TODO
    "model": None,      # TODO
    "input_dir": None,  # TODO
    "stage_dir": None,  # TODO
}

assert key_meanings["stage"].startswith("which ACORN")
assert "class" in key_meanings["model"]
assert "input" in key_meanings["input_dir"]
assert "checkpoints" in key_meanings["stage_dir"]

## 3. Train a one-step InteractionGNN

The supplied template uses one graph iteration, hidden width 16, and ten epochs. The important reliability settings are CPU acceleration, zero worker processes, and disabled W&B logging. The resolved config is written only to this notebook's temporary workspace.

In [ ]:
train_template = ROOT / "configs" / "02_interaction_gnn_train.yaml"
train_config = yaml.safe_load(train_template.read_text())
train_config["input_dir"] = None       # TODO: str(DATA)
train_config["stage_dir"] = None       # TODO: str(STAGE_DIR)
train_config["accelerator"] = None     # TODO: "cpu"
train_config["num_workers"] = None     # TODO: [0, 0, 0]
train_config["log_wandb"] = None       # TODO: False

assert train_config["accelerator"] == "cpu"
assert train_config["num_workers"] == [0, 0, 0]
assert train_config["log_wandb"] is False
assert train_config["n_graph_iters"] == 1

train_path = RUNTIME / "train.yaml"
train_path.write_text(yaml.safe_dump(train_config, sort_keys=False))
run_acorn("train", train_path)

## 4. A checkpoint connects training to inference

Lightning stores learned parameters and hyperparameters in `.ckpt` files. ACORN inference searches `stage_dir` for the newest checkpoint when none is supplied explicitly. CSV logs make the short loss history inspectable without an online service.

In [ ]:
checkpoints = None  # TODO: sorted .ckpt files in STAGE_DIR / "artifacts"
checkpoint = None   # TODO: select the first checkpoint

metric_paths = sorted(STAGE_DIR.rglob("metrics.csv"))
history = pd.read_csv(metric_paths[0])
print("checkpoint:", checkpoint.name)
print(history.dropna(subset=["val_loss"])[["epoch", "val_loss"]].tail())
assert checkpoint.suffix == ".ckpt"
assert "val_loss" in history.columns

## 5. Inference writes scored events

Training changes parameters; inference keeps them fixed, calculates one probability per candidate edge, stores it as `edge_scores`, appends the stage configuration, and writes new `.pyg` files under `stage_dir`.

In [ ]:
infer_config = yaml.safe_load((ROOT / "configs" / "02_interaction_gnn_infer.yaml").read_text())
infer_config["input_dir"] = str(DATA)
infer_config["stage_dir"] = str(STAGE_DIR)
infer_path = RUNTIME / "infer.yaml"
infer_path.write_text(yaml.safe_dump(infer_config, sort_keys=False))
run_acorn("infer", infer_path)

scored_paths = None  # TODO: sorted .pyg files in STAGE_DIR / "testset"
scored_event = None  # TODO: load the first scored event on CPU

print("scores:", scored_event.edge_scores)
assert scored_event.edge_scores.shape == scored_event.edge_y.shape
assert torch.isfinite(scored_event.edge_scores).all()
assert len(scored_event.config) >= 1

## 6. Evaluation applies an operating point

`acorn eval` reads scored events, applies the requested target-track selection, and produces configured plots. A score cut converts probabilities into decisions. Edge efficiency is the fraction of truth edges retained; edge purity is the fraction of selected edges that are true.

In [ ]:
eval_config = yaml.safe_load((ROOT / "configs" / "02_interaction_gnn_eval.yaml").read_text())
eval_config["input_dir"] = str(DATA)
eval_config["stage_dir"] = str(STAGE_DIR)
eval_path = RUNTIME / "eval.yaml"
eval_path.write_text(yaml.safe_dump(eval_config, sort_keys=False))
run_acorn("eval", eval_path)

In [ ]:
score_cut = eval_config["score_cut"]
predicted = None       # TODO: edge_scores >= score_cut
true_positive = None   # TODO: selected edges that are also true
selected = None        # TODO: number of selected edges
truth = None           # TODO: number of true edges
edge_efficiency = None # TODO: true_positive / truth
edge_purity = None     # TODO: true_positive / selected, or 0 if none

print(f"edge efficiency at {score_cut}: {edge_efficiency:.1%}")
print(f"edge purity at {score_cut}: {edge_purity:.1%}")
assert 0.0 <= edge_efficiency <= 1.0
assert 0.0 <= edge_purity <= 1.0

## 7. The complete data flow

```text
tutorial_data/edge_classifier/*.pyg
        │  acorn train + train.yaml
        ▼
stage_dir/artifacts/*.ckpt + CSV logs
        │  acorn infer + infer.yaml
        ▼
stage_dir/{trainset,valset,testset}/*.pyg with edge_scores
        │  acorn eval + eval.yaml
        ▼
evaluation plots and operating-point metrics
```

## 8. Recap and next step

You can now explain how YAML selects a stage and model, why input and output directories differ, why training/inference/evaluation are separate operations, and how a checkpoint connects them. Tutorial 03 implements the model class that sits inside this workflow.